# LightGPT Colab — Lightweight, low-RAM demo

This Colab notebook demonstrates how to run LightGPT in a low-memory environment, export ONNX/NPZ artifacts, run a tiny NumPy runtime (Underkill), and optionally package and publish to PyPI.

Follow the cells in order. The notebook clones the repository, installs the package in editable mode, and runs lightweight demos optimized for Colab.

In [ ]:
# Clone the repo (force re-clone for clean state)
import os, subprocess, sys
import shutil

REPO='https://github.com/Maor-404/LightGPT.git'
REPO_DIR = 'LightGPT'
BASE_PATH = '/content' # Ensure operations start from /content

# Change to base path first to ensure cloning happens in /content
os.chdir(BASE_PATH)

if os.path.exists(REPO_DIR):
    print(f'Removing existing {REPO_DIR} directory...')
    shutil.rmtree(REPO_DIR)

print(f'Cloning {REPO_DIR}...')
subprocess.run(['git','clone',REPO], check=True)

# Now change into the cloned directory, which should be /content/LightGPT
os.chdir(os.path.join(BASE_PATH, REPO_DIR))
print('Installing editable package and runtime deps...')
# Editable install and requirements (quiet)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)
print('Setup complete')

Removing existing LightGPT directory...
Cloning LightGPT...
Installing editable package and runtime deps...


In [ ]:
# Quick environment check: Python, PyTorch, CUDA (if available)
import sys, torch
print('Python', sys.version)
print('PyTorch', getattr(torch,'__version__',None))
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# Run a quick CPU demo (normal mode)
import subprocess, sys
print('Running examples/run_demo.py (normal)')
subprocess.run([sys.executable,'examples/run_demo.py','--mode','normal','--prompt','Hello Colab'], check=False)

## Tiny NumPy runtime (Underkill)
The `TinyLightGPT` runtime runs with pure NumPy and small NPZ weight files. Use this on very constrained environments. The next cells create a tiny checkpoint, export NPZ weights, and run the tiny runtime demo.

In [ ]:
import sys
import os

# Ensure the current directory (which should be LightGPT after cell bbb2e067)
# and potentially its 'src' subdirectory are in sys.path to allow importing lightgpt.
# This explicitly addresses ModuleNotFoundError if the editable install didn't properly register.
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# Check for 'src' layout common in modern Python projects
src_dir = os.path.join(current_dir, 'src')
if os.path.exists(src_dir) and src_dir not in sys.path:
    sys.path.insert(0, src_dir)

In [ ]:
# Create a tiny 'underkill' checkpoint and export NPZ weights
from lightgpt import LightGPT
import torch, subprocess, sys, os
m = LightGPT(mode='underkill', max_seq_len=32)
torch.save({'model_state_dict': m.state_dict()}, 'lightgpt.chkpt')
print('Saved lightgpt.chkpt')
# Export NPZ using provided script
subprocess.run([sys.executable,'scripts/export_npz.py','lightgpt.chkpt','lightgpt_weights.npz'], check=True)
print('Exported lightgpt_weights.npz')

In [ ]:
# Run tiny runtime demo (greedy generation)
from lightgpt.tiny_runtime import TinyLightGPT
from lightgpt.model import LightGPT
rt = TinyLightGPT('lightgpt_weights.npz')
prompt = LightGPT.small_vocab_tokenize('Hello from Colab tiny runtime')
out = rt.generate(prompt, max_new_tokens=12)
print('Output ids (tiny runtime):', out[:40])

## ONNX export, quantize, and run (CPU)
The following exports a dynamic-axes ONNX file, optionally quantizes it to INT8 using ONNX Runtime tools, and runs an ONNX inference example. ONNX export may be slower; use the tiny runtime for minimal RAM consumption.

In [ ]:
# Export ONNX (dynamic axes)
import subprocess, sys
print('Exporting ONNX...')
subprocess.run([sys.executable,'examples/export_onnx.py','--out','lightgpt.onnx','--mode','normal','--seq','64'], check=True)
print('Export finished')
# Quantize ONNX (optional)
try:
    subprocess.run([sys.executable,'examples/quantize_onnx.py','lightgpt.onnx'], check=True)
    print('Quantized ONNX saved as lightgpt.quant.onnx')
except Exception as e:
    print('Quantization failed or skipped:', e)

In [ ]:
# Run ONNX Runtime inference demo (quantized if available)
import os, subprocess, sys
if os.path.exists('lightgpt.quant.onnx'):
    model_path = 'lightgpt.quant.onnx'
elif os.path.exists('lightgpt.onnx'):
    model_path = 'lightgpt.onnx'
else:
    model_path = None
if model_path:
    print('Running ONNX demo on', model_path)
    subprocess.run([sys.executable,'examples/run_onnx.py',model_path,'--prompt','Hello Colab','--intra','2','--inter','1'], check=False)
else:
    print('No ONNX model found; skip ONNX run')

## Publish to PyPI (optional)
If you want to publish this package to PyPI from Colab, provide your PyPI API token securely.
**DO NOT** paste your raw API token into notebook cells. Instead, set it in Colab secrets or use environment variables. The recommended way is to set `TWINE_USERNAME='__token__'` and `TWINE_PASSWORD` to your API token.

In [ ]:
# Build wheel and optionally upload to PyPI using TWINE (requires TWINE_USERNAME/TWINE_PASSWORD env set)
# Uncomment and run only if you know what you're doing and have set secure env vars
# !python -m pip install --upgrade build twine
# !python -m build
# !twine upload --non-interactive -u __token__ -p $TWINE_PASSWORD dist/*
print('Build/upload commands are provided but commented out for safety.')

## Final notes
- Prefer the `underkill` tiny runtime when RAM is the constraint.
- Use ONNX quantized models for faster CPU inference when available.
- For CI reproducibility, consider using the included GitHub Actions workflow or the Docker option.

If anything fails, copy the error output and I will help debug. Enjoy LightGPT on Colab!